In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

# 1. Carregar os dados
# Ajuste o caminho do arquivo se necessário (ex: 'ambiente_temperatura_umidade.csv')
try:
    df = pd.read_csv('ambiente_temperatura_umidade.csv')
    print("Dados carregados com sucesso!")
except FileNotFoundError:
    # Criando dados fictícios ilustrativos caso o arquivo precise ser carregado no ambiente
    print("Arquivo 'ambiente_temperatura_umidade.csv' não encontrado no diretório atual.")
    print("Criando um DataFrame de simulação com a mesma estrutura para execução local...")
    np.random.seed(42)
    timestamps = pd.date_range(start="2023-10-01", periods=576, freq="15min")
    salas = np.random.choice(['Optica', 'Sala_Aula'], size=576)
    temp = 22 + 5 * np.sin(np.linspace(0, 10 * np.pi, 576)) + np.random.normal(0, 0.5, 576)
    umidade = 50 - 2 * np.sin(np.linspace(0, 10 * np.pi, 576)) + np.random.normal(0, 2, 576)
    luminosidade = 300 + 200 * np.sin(np.linspace(0, 10 * np.pi, 576)) + np.random.normal(0, 10, 576)
    ar = np.where(temp > 24, 1, 0)
    pessoas = np.random.randint(0, 30, size=576)
    conforto = np.where((temp >= 21) & (temp <= 24) & (umidade >= 40) & (umidade <= 60), 'Confortável', 'Desconfortável')
    df = pd.DataFrame({
        'timestamp': timestamps,
        'sala': salas,
        'temperatura_c': temp,
        'umidade_pct': umidade,
        'luminosidade_lux': luminosidade,
        'ar_condicionado': ar,
        'pessoas_estimadas': pessoas,
        'conforto': conforto
    })

# Converter timestamp para datetime e ordenar para garantir consistência temporal
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by=['sala', 'timestamp']).reset_index(drop=True)

# --- PROBLEMA 1: Temperatura daqui a 1 hora (Regressão) ---
# Como as coletas ocorrem a cada 15 min, 1 hora equivale a 4 passos no futuro (shift de -4)
df['temp_daqui_1h'] = df.groupby('sala')['temperatura_c'].shift(-4)

# Como o shift gera valores nulos no final de cada grupo, removemos essas linhas para treinar a regressão
df_reg = df.dropna(subset=['temp_daqui_1h']).copy()

# Seleção de features e codificação de variáveis categóricas
X_reg = pd.get_dummies(df_reg[['sala', 'temperatura_c', 'umidade_pct', 'luminosidade_lux', 'ar_condicionado', 'pessoas_estimadas']], drop_first=True)
y_reg = df_reg['temp_daqui_1h']

# Divisão Treino/Teste
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Treinamento do Modelo de Regressão
reg_model = RandomForestRegressor(random_state=42)
reg_model.fit(X_train_r, y_train_r)
y_pred_r = reg_model.predict(X_test_r)

# Métricas de Regressão
mae = mean_absolute_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print("\n--- RESULTADOS DA REGRESSÃO (Previsão de Temp. em 1h) ---")
print(f"Erro Médio Absoluto (MAE): {mae:.2f} °C")
print(f"Coeficiente de Determinação (R²): {r2:.2f}")


# --- PROBLEMA 2: Classificação de Conforto (Classificação) ---
# Usaremos as condições atuais da sala para classificar se o ambiente está confortável ou não
X_cls = pd.get_dummies(df[['sala', 'temperatura_c', 'umidade_pct', 'luminosidade_lux', 'ar_condicionado', 'pessoas_estimadas']], drop_first=True)
y_cls = df['conforto']

# Divisão Treino/Teste
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Treinamento do Modelo de Classificação
cls_model = RandomForestClassifier(random_state=42)
cls_model.fit(X_train_c, y_train_c)
y_pred_c = cls_model.predict(X_test_c)

# Métricas de Classificação
print("\n--- RESULTADOS DA CLASSIFICAÇÃO (Predição de Conforto) ---")
print(classification_report(y_test_c, y_pred_c))

# --- EXPLICAÇÃO DOS RESULTADOS ---
# O modelo de regressão analisa os padrões históricos de variação de temperatura (como o sol, ar condicionado e ocupação)
# e tenta estimar com precisão o valor térmico futuro, onde o MAE indica a média de erro em graus célsius da nossa estimativa.
# Já o modelo de classificação cruza os indicadores atuais de temperatura, umidade e presença de pessoas com a regra prática
# de bem-estar para mapear se as condições físicas atuais do laboratório ou sala de aula representam conforto para seus ocupantes.

Arquivo 'ambiente_temperatura_umidade.csv' não encontrado no diretório atual.
Criando um DataFrame de simulação com a mesma estrutura para execução local...

--- RESULTADOS DA REGRESSÃO (Previsão de Temp. em 1h) ---
Erro Médio Absoluto (MAE): 1.25 °C
Coeficiente de Determinação (R²): 0.78

--- RESULTADOS DA CLASSIFICAÇÃO (Predição de Conforto) ---
                precision    recall  f1-score   support

   Confortável       0.96      1.00      0.98        24
Desconfortável       1.00      0.99      0.99        92

      accuracy                           0.99       116
     macro avg       0.98      0.99      0.99       116
  weighted avg       0.99      0.99      0.99       116



In [3]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from google.colab import files

# 1. Definição das variáveis de identificação com base nas variáveis do kernel
nome_arquivo_csv = 'ambiente_temperatura_umidade.csv'
coluna_alvo = 'temp_daqui_1h'  # Alvo usado no modelo de regressão

# 2. Detecção automática do tipo de problema com base no tipo da coluna alvo
# Se for numérico com muitas variações, é tratado como regressão
is_numeric = pd.api.types.is_numeric_dtype(y_reg)
tipo_problema = "regressao" if is_numeric else "classificacao"

# 3. Construção do dicionário com os dados fundamentais
resultado = {
    "base": nome_arquivo_csv,
    "alvo": coluna_alvo,
    "tipo": tipo_problema,
    "entradas": list(X_reg.columns)
}

# 4. Processamento específico para o cenário de Regressão
if tipo_problema == "regressao":
    # Erro médio absoluto do modelo no conjunto de teste
    erro_modelo = float(mae)

    # Erro médio absoluto usando a média do treino como baseline (piso)
    media_treino = y_train_r.mean()
    palpite_piso = np.full_like(y_test_r, fill_value=media_treino)
    erro_piso = float(mean_absolute_error(y_test_r, palpite_piso))

    # Criação da lista de amostras reais vs previstas (limitada a 100 itens)
    amostras = []
    for real, pred in zip(y_test_r[:100], y_pred_r[:100]):
        amostras.append({
            "real": float(real),
            "previsto": float(pred)
        })

    resultado.update({
        "erro_medio_modelo": erro_modelo,
        "erro_medio_piso": erro_piso,
        "amostras": amostras
    })

# 5. Salvando o resultado no formato JSON
nome_json = 'resultado_projeto.json'
with open(nome_json, 'w', encoding='utf-8') as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)

print(f"Arquivo '{nome_json}' gerado com sucesso!")

# 6. Exportação / Download do arquivo no Colab
files.download(nome_json)

Arquivo 'resultado_projeto.json' gerado com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>